In [ ]:
!pip install -Uqq ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 26.7 MB/s eta 0:00:00


In [ ]:
from ddgs import DDGS
from fastai.vision.all import *

In [ ]:
from pathlib import Path

path = Path("/content/bears")

classes = ["grizzly", "black", "teddy"]

for cls in classes:
    (path / cls).mkdir(parents=True, exist_ok=True)

path

Path('/content/bears')

In [ ]:
import requests

def download_images(query, folder, n=150):
    results = DDGS().images(
        query=query,
        max_results=n
    )

    downloaded = 0

    for i, result in enumerate(results):
        try:
            url = result["image"]
            response = requests.get(url, timeout=10)

            if response.status_code == 200:
                ext = ".jpg"
                file_path = folder / f"{i}{ext}"
                file_path.write_bytes(response.content)
                downloaded += 1

        except Exception:
            pass

    print(f"{query}: downloaded {downloaded} images")

In [ ]:
download_images("grizzly bear", path/"grizzly", 150)
download_images("black bear", path/"black", 150)
download_images("teddy bear", path/"teddy", 150)

grizzly bear: downloaded 34 images
black bear: downloaded 27 images
teddy bear: downloaded 34 images


In [ ]:
for cls in classes:
    files = get_image_files(path / cls)
    failed = verify_images(files)
    print(cls, len(failed))

grizzly 1
black 1
teddy 0


In [ ]:
for cls in classes:
    files = get_image_files(path / cls)
    failed = verify_images(files)

    for f in failed:
        print("Deleting:", f)
        f.unlink()

Deleting: /content/bears/grizzly/20.jpg
Deleting: /content/bears/black/24.jpg


In [ ]:
for cls in classes:
    files = get_image_files(path / cls)
    failed = verify_images(files)
    print(cls, "images:", len(files), "| invalid:", len(failed))

grizzly images: 34 | invalid: 0
black images: 26 | invalid: 0
teddy images: 35 | invalid: 0


In [ ]:
from fastai.vision.all import *

for cls in classes:
    print(f"\n{cls.upper()}")

    files = get_image_files(path / cls)[:6]
    imgs = [PILImage.create(f) for f in files]

    show_images(imgs, nrows=2, ncols=3, figsize=(9, 6))

In [ ]:
bears = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=Resize(128)
)

dls = bears.dataloaders(path, bs=8)

In [ ]:
dls.show_batch(max_n=9)

In [ ]:
learn = vision_learner(
    dls,
    resnet18,
    metrics=error_rate
)
learn.fine_tune(3)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 138MB/s]


epoch,train_loss,valid_loss,error_rate,time
0,1.607660,0.356172,0.052632,00:07


epoch,train_loss,valid_loss,error_rate,time
0,0.359107,0.260202,0.052632,00:06
1,0.270030,0.019810,0.000000,00:06
2,0.212019,0.020339,0.000000,00:05
